In [39]:
import math
import re

import pandas as pd
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
pd.set_option('display.max_colwidth', None)

In [40]:
sentences = [
    '이 영화 정말 좋다',
    '이 영화 너무 지루하다',
    '배우 연기가 감동적이다',
    '전개는 보통이지만 마지막이 재미있다',
]

In [41]:
from pathlib import Path
from gensim.models import Word2Vec
import sys
import pandas as pd

# 프로젝트 루트 경로
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

from preprocessing import preprocess_dataframe, build_vocab

# ===== Step 1: Daum 리뷰 데이터 로드 =====
# GitHub에서 직접 다운로드
DEFAULT_URL = "https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv"

print('Daum 리뷰 데이터 다운로드 중...')
try:
    df = pd.read_csv(DEFAULT_URL)
    print(f'다운로드 완료: {len(df)}개 샘플')
    print(f'컬럼: {list(df.columns)}')
except Exception as e:
    print(f'다운로드 실패: {e}')
    raise

# 컬럼 정규화
if "review" in df.columns:
    review_col = "review"
else:
    review_col = df.columns[0]

if "rating(1~10점)" in df.columns:
    rating_col = "rating(1~10점)"
elif "rating" in df.columns:
    rating_col = "rating"
else:
    rating_col = df.columns[1]

df = df[[review_col, rating_col]].copy()
df.columns = ["review", "rating"]

# 감정 레이블 생성
df["sentiment"] = (df["rating"] >= 6).astype(int)
train_df = df[["review", "sentiment"]].copy()
print(f'긍정: {(train_df["sentiment"]==1).sum()}, 부정: {(train_df["sentiment"]==0).sum()}')

# ===== Step 2: 전처리 및 tokenization =====
print('\n데이터 전처리 중...')
df_processed, vocab = preprocess_dataframe(
    train_df, 
    text_col='review', 
    label_col='sentiment',
    max_len=30,
    min_freq=2
)
word_to_idx = vocab
idx_to_word = {v: k for k, v in vocab.items()}
print(f'Vocab 크기: {len(vocab)}')

# ===== Step 3: Word2Vec 모델 학습 =====
print('\nWord2Vec 모델 학습 중... (1-2분 소요)')
tokenized_sentences = []
for tokens in df_processed['token_ids']:
    if len(tokens) > 0:
        words = [idx_to_word.get(idx, '<unk>') for idx in tokens if idx in idx_to_word]
        tokenized_sentences.append(words)

# Word2Vec 학습
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=5,
    seed=42
)
print(f'Word2Vec 모델 학습 완료 (vector_size={w2v_model.vector_size})')

# ===== Step 4: toy_vocab을 Word2Vec embedding으로 업데이트 =====
print('\ntoy_vocab을 Word2Vec embedding으로 업데이트 중...')

toy_vocab = {}
for word in vocab:
    if word in w2v_model.wv:
        toy_vocab[word] = torch.from_numpy(w2v_model.wv[word]).float()
    else:
        toy_vocab[word] = torch.zeros(w2v_model.vector_size).float()

print(f'{len(toy_vocab)}개 단어의 embedding 준비 완료')
print(f'embedding 차원: {w2v_model.vector_size}')
print('\n샘플 단어들 (처음 5개 차원):')

sample_words = ['좋다', '지루하다', '재미있다', '영화', '배우', '감동', '재미']
for word in sample_words:
    if word in toy_vocab:
        vec = toy_vocab[word][:5]
        print(f'{word}: {[round(v, 3) for v in vec.tolist()]}...')

# ===== Step 5: query를 100차원으로 업데이트 =====
positive_query = torch.ones(w2v_model.vector_size) * 0.1
negative_query = torch.ones(w2v_model.vector_size) * -0.1

print(f'\nQuery 벡터 업데이트 완료')
print(f'positive_query 크기: {positive_query.shape}')
print(f'negative_query 크기: {negative_query.shape}')
print(f'\n주의: toy_vocab은 이제 실제 Word2Vec embedding입니다.')


Daum 리뷰 데이터 다운로드 중...
다운로드 완료: 14725개 샘플
컬럼: ['review', 'rating', 'date', 'title']
긍정: 11179, 부정: 3546

데이터 전처리 중...
Vocab 크기: 14928

Word2Vec 모델 학습 중... (1-2분 소요)
Word2Vec 모델 학습 완료 (vector_size=100)

toy_vocab을 Word2Vec embedding으로 업데이트 중...
14928개 단어의 embedding 준비 완료
embedding 차원: 100

샘플 단어들 (처음 5개 차원):
좋다: [0.044, 0.068, 0.247, -0.244, 0.024]...
지루하다: [0.011, 0.017, 0.058, -0.049, 0.005]...
재미있다: [0.031, 0.023, 0.103, -0.124, 0.013]...
영화: [0.013, 0.311, 0.908, -0.73, 0.031]...
배우: [0.086, 0.091, 0.412, -0.429, 0.037]...
감동: [0.082, 0.127, 0.556, -0.53, 0.021]...
재미: [0.071, 0.127, 0.523, -0.472, 0.037]...

Query 벡터 업데이트 완료
positive_query 크기: torch.Size([100])
negative_query 크기: torch.Size([100])

주의: toy_vocab은 이제 실제 Word2Vec embedding입니다.


In [42]:
# Removed 2D manual toy embeddings to avoid confusion.
# This notebook now uses the trained Word2Vec embeddings in `toy_vocab` exclusively.
toy_vocab_2d = None
toy_positive_query = None
toy_negative_query = None
print('2D toy embeddings removed. Use the trained Word2Vec `toy_vocab` by running the Word2Vec training cell if needed.')

2D toy embeddings removed. Use the trained Word2Vec `toy_vocab` by running the Word2Vec training cell if needed.


In [43]:
def simple_tokenize(text: str):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    return text.split()

def get_token_vectors(tokens, vocab):
    # Determine expected embedding dimension from vocab (if possible)
    dim = None
    try:
        for v in vocab.values():
            if hasattr(v, 'shape'):
                dim = v.shape[0]
                break
    except Exception:
        dim = None

    if dim is None:
        dim = 100

    vectors = []
    for token in tokens:
        vec = vocab.get(token, None)
        if vec is None:
            vectors.append(torch.zeros(dim))
        else:
            # ensure tensor and correct dim
            if not hasattr(vec, 'shape'):
                vec = torch.tensor(vec).float()
            if vec.shape[0] != dim:
                raise RuntimeError(f"Embedding dimension mismatch for token '{token}': {vec.shape[0]} vs expected {dim}")
            vectors.append(vec)

    return torch.stack(vectors) if vectors else torch.empty(0, dim)

def attention_with_query(tokens, query, vocab):
    """
    Attention 계산 함수.
    
    입력:
    - tokens: 문장을 토큰으로 나눈 리스트, 예: ['이', '영화', '정말', '좋다']
    - query: 지금 찾고 싶은 의미 벡터, 예: [1.0, 1.0] (positive)
    - vocab: 각 토큰의 embedding 벡터를 담은 딕셔너리
    
    출력:
    - frame: 각 토큰의 key, score, weight, value를 담은 DataFrame
    - context: query와 문장의 attention 결과로 만든 요약 벡터
    """
    
    # ===== Step 1: 각 토큰을 벡터로 바꾼다 =====
    # tokens를 vocab에서 찾아서 벡터로 변환한다.
    # 예: '좋다' -> [1.2, 1.0, ...] (100차원)
    key = get_token_vectors(tokens, vocab)
    
    # 간단한 예제에서는 key와 value를 같게 둔다.
    # (실제 모델에서는 key와 value를 만드는 다른 layer가 있다)
    value = key
    
    # 토큰이 없으면 빈 결과를 반환한다
    if key.numel() == 0:
        return pd.DataFrame(), torch.tensor([])
    
    # ===== Step 2: query와 key의 유사도를 계산한다 (score) =====
    # score = query · key / sqrt(d_k)
    # dot product는 두 벡터가 같은 방향일수록 크다.
    d_k = key.size(-1)  # embedding 차원
    scores = torch.matmul(key, query) / math.sqrt(d_k)
    # scores shape: [토큰 개수]
    
    # ===== Step 3: score를 확률처럼 바꾼다 (weight) =====
    # softmax를 사용해서 score들을 0~1 사이의 값으로 정규화한다.
    # 모든 weight의 합은 1이 된다.
    # score가 높을수록 weight도 크다.
    weights = F.softmax(scores, dim=0)
    # weights shape: [토큰 개수]
    
    # ===== Step 4: weight를 이용해서 value들을 섞는다 (context) =====
    # context = sum(weight_i * value_i)
    # 이것이 attention의 핵심이다.
    # weight가 높은 토큰의 value가 더 많이 더해진다.
    context = torch.sum(weights.unsqueeze(-1) * value, dim=0)
    # context shape: [embedding 차원]
    
    # ===== 결과를 DataFrame으로 정리한다 =====
    # 각 토큰에 대해 score, weight를 표로 만든다
    # (100차원 embedding은 표로 출력하기 어려우므로 요약만 함)
    frame = pd.DataFrame({
        'token': tokens,
        'score': scores.tolist(),
        'weight': weights.tolist(),
    })
    
    return frame, context

In [44]:
sentence = sentences[0]
tokens = simple_tokenize(sentence)

# 강제: 항상 100차원 Word2Vec(`toy_vocab`) 사용
if 'toy_vocab' not in globals() or not toy_vocab:
    raise RuntimeError("Word2Vec 임베딩이 없습니다. 먼저 '2-1. 실제 Word2Vec Embedding 학습하기' 셀을 실행하세요.")

# 학습된 쿼리가 있으면 우선 사용, 없으면 기본 positive_query 사용
if 'learned_positive_query' in globals():
    query_to_use = learned_positive_query
elif 'positive_query' in globals():
    query_to_use = positive_query
else:
    # 안전장치: toy_vocab의 차원에 맞춰 기본 쿼리 생성
    first_vec = next(iter(toy_vocab.values()))
    query_to_use = torch.ones(first_vec.shape[0]) * 0.1

frame, context = attention_with_query(tokens, query_to_use, toy_vocab)

print('sentence:', sentence)
print('tokens:', tokens)
print('using_vocab: toy_vocab (Word2Vec)')
print('query_shape:', list(query_to_use.shape))
print('context:', context.tolist())
display(frame.sort_values('weight', ascending=False).reset_index(drop=True))

sentence: 이 영화 정말 좋다
tokens: ['이', '영화', '정말', '좋다']
using_vocab: toy_vocab (Word2Vec)
query_shape: [100]
context: [0.014127153903245926, 0.09480303525924683, 0.28879183530807495, -0.2435852438211441, 0.01385796070098877, 0.3925956189632416, -0.1772213727235794, 0.3116103708744049, -0.150526762008667, -0.4048519730567932, -0.056927286088466644, 0.2775823473930359, -0.1741824746131897, 0.33229103684425354, 0.24467132985591888, -0.29168418049812317, -0.20173360407352448, -0.3598848283290863, -0.03875119239091873, -0.2244887351989746, 0.07787491381168365, 0.31160229444503784, 0.06828926503658295, 0.8662656545639038, 0.25364744663238525, 0.23329570889472961, -0.5210875868797302, 0.09774772822856903, -0.15085437893867493, -0.11790694296360016, -0.13561579585075378, -0.2176285833120346, -0.10276313871145248, 0.19137556850910187, 0.2139878273010254, -0.22736743092536926, 0.008588471449911594, -0.12386974692344666, -0.2889629900455475, 0.22675715386867523, 0.19329039752483368, -0.0820589959621

,token,score,weight
0,영화,0.000346,0.250261
1,이,0.000000,0.250175
2,정말,0.000000,0.250175
3,좋다,-0.003146,0.249389


In [46]:
# 긍정 리뷰와 부정 리뷰의 embedding 평균을 계산
print('데이터에서 긍정/부정 query 추출 중...')

positive_indices = (train_df['sentiment'] == 1).values
negative_indices = (train_df['sentiment'] == 0).values

# 각 리뷰의 토큰 embedding의 평균을 구한다
positive_vecs = []
negative_vecs = []

for idx, token_ids in enumerate(df_processed['token_ids']):
    if len(token_ids) == 0:
        continue
    
    # 토큰 embedding 평균
    vecs = [toy_vocab.get(idx_to_word.get(tid, '<unk>'), torch.zeros(100)) for tid in token_ids if tid in idx_to_word]
    if vecs:
        avg_vec = torch.stack(vecs).mean(dim=0)
        if train_df.iloc[idx]['sentiment'] == 1:
            positive_vecs.append(avg_vec)
        else:
            negative_vecs.append(avg_vec)

# 긍정/부정 쿼리 계산
learned_positive_query = torch.stack(positive_vecs[:1000]).mean(dim=0) if positive_vecs else torch.zeros(100)
learned_negative_query = torch.stack(negative_vecs[:1000]).mean(dim=0) if negative_vecs else torch.zeros(100)

print(f'긍정 query 샘플: {learned_positive_query[:5].tolist()}...')
print(f'부정 query 샘플: {learned_negative_query[:5].tolist()}...')


데이터에서 긍정/부정 query 추출 중...
긍정 query 샘플: [-0.009408576413989067, 0.17944574356079102, 0.47384244203567505, -0.35490483045578003, -0.011808928102254868]...
부정 query 샘플: [-0.013257347047328949, 0.18797770142555237, 0.48986876010894775, -0.3615303337574005, -0.014847717247903347]...


In [50]:
sentence = sentences[0]
tokens = simple_tokenize(sentence)

# 강제: 항상 100차원 Word2Vec(`toy_vocab`) 사용
if 'toy_vocab' not in globals() or not toy_vocab:
    raise RuntimeError("Word2Vec 임베딩이 없습니다. 먼저 '2-1. 실제 Word2Vec Embedding 학습하기' 셀을 실행하세요.")

# 학습된 쿼리가 있으면 우선 사용, 없으면 기본 positive_query 사용
if 'learned_positive_query' in globals():
    query_to_use = learned_positive_query
elif 'positive_query' in globals():
    query_to_use = positive_query
else:
    # 안전장치: toy_vocab의 차원에 맞춰 기본 쿼리 생성
    first_vec = next(iter(toy_vocab.values()))
    query_to_use = torch.ones(first_vec.shape[0]) * 0.1

frame, context = attention_with_query(tokens, query_to_use, toy_vocab)

print('sentence:', sentence)
print('tokens:', tokens)
print('using_vocab: toy_vocab (Word2Vec)')
print('query_shape:', list(query_to_use.shape))
print('context:', context.tolist())
display(frame.sort_values('weight', ascending=False).reset_index(drop=True))

sentence: 이 영화 정말 좋다
tokens: ['이', '영화', '정말', '좋다']
using_vocab: toy_vocab (Word2Vec)
query_shape: [100]
context: [0.01532084308564663, 0.25994789600372314, 0.7642778158187866, -0.6199100613594055, 0.02792203612625599, 0.9857203364372253, -0.48872867226600647, 0.7743597626686096, -0.415312796831131, -1.0507352352142334, -0.11512776464223862, 0.7169516086578369, -0.4343683123588562, 0.8298989534378052, 0.6444382071495056, -0.7169171571731567, -0.536019504070282, -0.8991733193397522, -0.07830703258514404, -0.549079954624176, 0.20293089747428894, 0.7538878917694092, 0.15682262182235718, 2.1419405937194824, 0.663688063621521, 0.59263014793396, -1.2961177825927734, 0.24094854295253754, -0.37409406900405884, -0.29250437021255493, -0.3462030291557312, -0.5410998463630676, -0.27769893407821655, 0.46660399436950684, 0.5372947454452515, -0.5626498460769653, 0.0200126301497221, -0.28064092993736267, -0.7159645557403564, 0.5915244221687317, 0.4773786664009094, -0.21195857226848602, 0.664153397083

,token,score,weight
0,영화,3.077809,0.810538
1,좋다,1.123256,0.114795
2,이,0.000000,0.037333
3,정말,0.000000,0.037333


In [47]:
test_sentence = '이 영화 정말 좋다'
tokens = simple_tokenize(test_sentence)

pos_frame, pos_context = attention_with_query(tokens, learned_positive_query, toy_vocab)
neg_frame, neg_context = attention_with_query(tokens, learned_negative_query, toy_vocab)

print(f'문장: {test_sentence}')
print(f'토큰: {tokens}')
print()
print('[긍정 query 사용]')
display(pos_frame.sort_values('weight', ascending=False).reset_index(drop=True))

print('\n[부정 query 사용]')
display(neg_frame.sort_values('weight', ascending=False).reset_index(drop=True))


문장: 이 영화 정말 좋다
토큰: ['이', '영화', '정말', '좋다']

[긍정 query 사용]


,token,score,weight
0,영화,3.077809,0.810538
1,좋다,1.123256,0.114795
2,이,0.000000,0.037333
3,정말,0.000000,0.037333



[부정 query 사용]


,token,score,weight
0,영화,3.126945,0.816478
1,좋다,1.139679,0.111914
2,이,0.000000,0.035804
3,정말,0.000000,0.035804


In [48]:
compare_sentence = '이 영화 정말 좋다 하지만 마지막이 별로다'
tokens = simple_tokenize(compare_sentence)

# 강제: 항상 100차원 Word2Vec(`toy_vocab`) 사용
if 'toy_vocab' not in globals() or not toy_vocab:
    raise RuntimeError("Word2Vec 임베딩이 없습니다. 먼저 '2-1. 실제 Word2Vec Embedding 학습하기' 셀을 실행하세요.")

# 학습된 쿼리가 있으면 우선 사용, 없으면 기본 쿼리 사용
pos_q = learned_positive_query if 'learned_positive_query' in globals() else (positive_query if 'positive_query' in globals() else None)
neg_q = learned_negative_query if 'learned_negative_query' in globals() else (negative_query if 'negative_query' in globals() else None)

if pos_q is None or neg_q is None:
    # 안전장치: toy_vocab 차원에 맞추어 기본값 생성
    dim = next(iter(toy_vocab.values())).shape[0]
    if pos_q is None:
        pos_q = torch.ones(dim) * 0.1
    if neg_q is None:
        neg_q = torch.ones(dim) * -0.1

pos_frame, pos_context = attention_with_query(tokens, pos_q, toy_vocab)
neg_frame, neg_context = attention_with_query(tokens, neg_q, toy_vocab)

print('sentence:', compare_sentence)
print('tokens:', tokens)
print()
print('[positive query]')
print('query_shape:', list(pos_q.shape))
print('context:', pos_context.tolist())
display(pos_frame.sort_values('weight', ascending=False).reset_index(drop=True))

print('[negative query]')
print('query_shape:', list(neg_q.shape))
print('context:', neg_context.tolist())
display(neg_frame.sort_values('weight', ascending=False).reset_index(drop=True))

sentence: 이 영화 정말 좋다 하지만 마지막이 별로다
tokens: ['이', '영화', '정말', '좋다', '하지만', '마지막이', '별로다']

[positive query]
query_shape: [100]
context: [0.01570739597082138, 0.22692526876926422, 0.6711305975914001, -0.5473526120185852, 0.025691090151667595, 0.8695944547653198, -0.4263708293437958, 0.6852248311042786, -0.3616674244403839, -0.9243446588516235, -0.10548561066389084, 0.6297892332077026, -0.38397538661956787, 0.7328616380691528, 0.5656395554542542, -0.6337751150131226, -0.470279723405838, -0.7938186526298523, -0.07155773788690567, -0.4858093857765198, 0.17813976109027863, 0.669274628162384, 0.14066579937934875, 1.894569993019104, 0.5834553837776184, 0.5226688385009766, -1.145788550376892, 0.2137521654367447, -0.3306724429130554, -0.25773435831069946, -0.30505087971687317, -0.4784317910671234, -0.2428063452243805, 0.41353321075439453, 0.47469639778137207, -0.4970172047615051, 0.017938978970050812, -0.2519460618495941, -0.6323915123939514, 0.5192580223083496, 0.4226868152618408, -0.18692599236

,token,score,weight
0,영화,3.077809,0.701462
1,좋다,1.123256,0.099346
2,마지막이,0.517582,0.054214
3,별로다,0.396874,0.048050
4,이,0.000000,0.032309
5,정말,0.000000,0.032309
6,하지만,0.000000,0.032309


[negative query]
query_shape: [100]
context: [0.015655135735869408, 0.229406476020813, 0.678155243396759, -0.5528196692466736, 0.025862639769911766, 0.8782297372817993, -0.4310589134693146, 0.6918976902961731, -0.36567234992980957, -0.9338338971138, -0.10620899498462677, 0.636271595954895, -0.3877400755882263, 0.7400843501091003, 0.5715523958206177, -0.6399228572845459, -0.47523167729377747, -0.8016524314880371, -0.07205553352832794, -0.49048930406570435, 0.1799887716770172, 0.6755825877189636, 0.14188252389431, 1.9129866361618042, 0.5894929766654968, 0.5279067754745483, -1.1569995880126953, 0.21580125391483307, -0.3339022994041443, -0.2602848410606384, -0.30812937021255493, -0.4831075072288513, -0.24542337656021118, 0.41747981309890747, 0.4793826639652252, -0.50187748670578, 0.018096575513482094, -0.2540910542011261, -0.6385834217071533, 0.5246509313583374, 0.4267747402191162, -0.18882004916667938, 0.5921906232833862, -0.49439534544944763, -0.42768341302871704, 0.7825428247451782, 0.3

,token,score,weight
0,영화,3.126945,0.710067
1,좋다,1.139679,0.097329
2,마지막이,0.524787,0.052626
3,별로다,0.402450,0.046566
4,이,0.000000,0.031138
5,정말,0.000000,0.031138
6,하지만,0.000000,0.031138


In [49]:
rows = []
for sentence in sentences:
    tokens = simple_tokenize(sentence)
    frame, context = attention_with_query(tokens, positive_query, toy_vocab)
    if len(frame) == 0:
        rows.append({
            'sentence': sentence,
            'top_token': None,
            'top_weight': None,
            'context_x': None,
            'context_y': None,
        })
        continue

    top_row = frame.sort_values('weight', ascending=False).iloc[0]
    rows.append({
        'sentence': sentence,
        'top_token': top_row['token'],
        'top_weight': round(float(top_row['weight']), 4),
        'context_x': round(float(context[0]), 4),
        'context_y': round(float(context[1]), 4),
    })

summary = pd.DataFrame(rows)
summary

,sentence,top_token,top_weight,context_x,context_y
0,이 영화 정말 좋다,영화,0.2503,0.0141,0.0948
1,이 영화 너무 지루하다,영화,0.2501,0.0059,0.0822
2,배우 연기가 감동적이다,감동적이다,0.3347,0.0584,0.0930
3,전개는 보통이지만 마지막이 재미있다,보통이지만,0.2503,0.0157,0.0151
